<a href="https://colab.research.google.com/github/Zeenith24/Flyrank-assessment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Scoring / Ranking (approached via Probability Classification).

Problem Statement: We need to score and rank web pages based on their urgency for a content refresh. The model will calculate a "probability of critical decline" for each page, allowing the SEO team to prioritize the top N most urgent pages rather than manually analyzing 30,000 URLs.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: The true need for a page rewrite (which is unobservable without human expert judgment).

Proxy: A derived binary label, is_critical_decline. Since we don't have a human-labeled "needed refresh" column, we will engineer a proxy target: 1 if the page is experiencing a severe downward trend (e.g., trend_direction == 'down' AND trend_pct <= -20% AND has meaningful search volume), and 0 otherwise. The model's probability output for this proxy will be our sorting score.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Offline Metric: Precision@K (e.g., Precision@20 or Precision@50). Because the content team will only pull the "top 20 flagged pages" each week, general accuracy doesn't matter as much as the density of true positives at the very top of the sorted list. We cannot afford False Positives in the top 20, as that wastes expensive editorial labor. (We will also track AUC-ROC for overall model health).

Content Action: The model scores all pages, sorts them descending by probability, and feeds the Top 20 to the content writer’s dashboard for manual auditing, intent-checking, and rewriting.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

# Load the W01 starter dataset
data_url = 'https://raw.githubusercontent.com/Zeenith24/Flyrank-assessment/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_url)

# Define the proxy target: "Critical Decline"
# Example threshold: Downward trend worse than -15% with at least some baseline traffic
# Note: Adjust the -15 threshold based on whether trend_pct is a decimal (-0.15) or whole number (-15)
df['target_critical_decline'] = (
    (df['trend_direction'] == 'down') &
    (df['trend_pct'] <= -15) &
    (df['impressions_90d'] > 100)
).astype(int)

# Select the unit of analysis and key features
# Unit of analysis: One row = One specific web page (content_id)
analysis_df = df[[
    'content_id',
    'content_age_days',
    'competition_level',
    'impressions_90d',
    'trend_pct',
    'target_critical_decline'
]]

print("Unit of analysis: One row equals one unique web page (content_id)")
analysis_df.head()

Unit of analysis: One row equals one unique web page (content_id)


,content_id,content_age_days,competition_level,impressions_90d,trend_pct,target_critical_decline
0,content_304f48230142,187,HIGH,3803,-41.4,1
1,content_a1fb4e703a9e,445,LOW,15320,-57.7,1
2,content_9aa793d4d895,141,LOW,12581,-60.9,1
3,content_331d6c4de07b,463,LOW,11751,-13.8,0
4,content_d99b7a2d90ca,263,LOW,19140,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Rationale: A simple fixed rule (e.g., IF trend_pct < -20% THEN flag for refresh) generates massive False Positives. A static rule treats a 20% drop on a highly volatile 14-day-old page exactly the same as a 20% drop on a stable, 3-year-old cornerstone page.

Machine Learning beats this because it can capture non-linear relationships. An ML model can weigh content_age_days, competition_level, engagement_rate, and impressions_90d simultaneously—learning that a traffic drop on a low-competition, low-engagement page might just be natural stabilization, whereas a smaller drop on a high-value, high-engagement page signals critical semantic decay that requires immediate action.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.